# CS336 Lecture 2 — CPU 分段练习

这是助手根据 Lecture 2 主题编写的小规模练习，**不是原课件逐行转换，也不是 A1 答案**。完整课堂内容、图片、原始结果请看 `lecture_02_recording.ipynb`。

所有实验默认 CPU、小张量。按顺序 Shift+Enter，可以改数字观察结果。依赖 torch、einops、ipykernel；推荐选择已有 CS336 WSL 虚拟环境。运行输出是你本机实际计算的结果，与录课中的硬件性能无关。


In [1]:
import torch
from torch import nn
from einops import rearrange, einsum, reduce
torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("这些练习使用 CPU；CUDA 可用：", torch.cuda.is_available())


PyTorch: 2.11.0+cu130
这些练习使用 CPU；CUDA 可用： False


/mnt/e/LLMStudy/LLM-study/cs336-a1/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. 张量维度与形状

试着修改 B、T、D。rank 是维度数量，shape 是每个维度的长度。

In [2]:
B, T, D = 2, 3, 4
x = torch.arange(B * T * D, dtype=torch.float32).reshape(B, T, D)
print(x)
print("shape:", x.shape, "rank:", x.ndim, "元素数:", x.numel())
assert x.ndim == 3


tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]])
shape: torch.Size([2, 3, 4]) rank: 3 元素数: 24


## 2. dtype 与内存

内存字节数 = 元素数 × 每个元素字节数。这里只分配小张量；原课件的 GPT-3 大矩阵改成纯算术估算。

In [3]:
for dtype in [torch.float32, torch.float16, torch.bfloat16, torch.int64]:
    x = torch.zeros(4, 8, dtype=dtype)
    print(dtype, x.element_size(), "bytes/element; total", x.numel() * x.element_size())
gpt3_matrix_bytes = (12288 * 4) * 12288 * 4
print("GPT-3 示例单矩阵估算 GiB:", gpt3_matrix_bytes / 1024**3)
for dtype in [torch.float16, torch.bfloat16, torch.float32]:
    print(dtype, torch.tensor([1e-8], dtype=dtype))


torch.float32 4 bytes/element; total 128
torch.float16 2 bytes/element; total 64
torch.bfloat16 2 bytes/element; total 64
torch.int64 8 bytes/element; total 256
GPT-3 示例单矩阵估算 GiB: 2.25
torch.float16 tensor([0.], dtype=torch.float16)
torch.bfloat16 tensor([1.0012e-08], dtype=torch.bfloat16)
torch.float32 tensor([1.0000e-08])


## 3. 矩阵乘法与转置

这里只做小矩阵乘法；每个 batch 内独立计算。先预测输出形状，再运行。

In [4]:
x = torch.randn(2, 3, 4)
y = torch.randn(2, 5, 4)
z = x @ y.transpose(-2, -1)
z_named = einsum(x, y, "batch query hidden, batch key hidden -> batch query key")
print("x:", x.shape, "y:", y.shape, "z:", z.shape)
torch.testing.assert_close(z, z_named)


x: torch.Size([2, 3, 4]) y: torch.Size([2, 5, 4]) z: torch.Size([2, 3, 5])


## 4. 广播

偏置只有最后一个维度，仍可应用于所有 batch 和位置。广播不等于你必须显式复制数据。

In [5]:
x = torch.zeros(2, 3, 4)
bias = torch.arange(4, dtype=torch.float32)
print(x + bias)
assert (x + bias).shape == (2, 3, 4)


tensor([[[0., 1., 2., 3.],
         [0., 1., 2., 3.],
         [0., 1., 2., 3.]],

        [[0., 1., 2., 3.],
         [0., 1., 2., 3.],
         [0., 1., 2., 3.]]])


## 5. rearrange 与 reduce

把特征维度拆成多个 head，再合回来；对指定维度求均值。

In [6]:
x = torch.arange(2 * 3 * 8, dtype=torch.float32).reshape(2, 3, 8)
heads = rearrange(x, "b t (h d) -> b h t d", h=2)
restored = rearrange(heads, "b h t d -> b t (h d)")
print("拆分后:", heads.shape)
print("每条序列的平均向量:", reduce(x, "b t d -> b d", "mean"))
torch.testing.assert_close(x, restored)


拆分后: torch.Size([2, 2, 3, 4])
每条序列的平均向量: tensor([[ 8.,  9., 10., 11., 12., 13., 14., 15.],
        [32., 33., 34., 35., 36., 37., 38., 39.]])


## 6. Parameter 与 Module

此处用内置层演示注册参数；A1 的 Linear 仍由你按题目要求实现。观察 weight 被纳入 parameters，而输入数据没有。

In [7]:
layer = nn.Linear(4, 6, bias=False)
x = torch.randn(2, 3, 4)
y = layer(x)
print("输出:", y.shape)
print([(name, tuple(p.shape)) for name, p in layer.named_parameters()])
print("参数数:", sum(p.numel() for p in layer.parameters()))
assert y.shape == (2, 3, 6)


输出: torch.Size([2, 3, 6])
[('weight', (6, 4))]
参数数: 24


## 7. 自动求导

y=w*x，loss=y²，数学上 dloss/dw=2*w*x²。backward 把梯度累加到 .grad。

In [8]:
w = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(2.0)
loss = (w * x) ** 2
loss.backward()
print("loss:", loss.item(), "gradient:", w.grad.item())
assert w.grad.item() == 24.0


loss: 36.0 gradient: 24.0


## 8. 小型训练循环

只演示更新流程，不是 A1 AdamW 实现。每一步：清梯度 → 前向 → 损失 → 反向 → 更新。

In [9]:
torch.manual_seed(42)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
x = torch.linspace(-1, 1, 32).reshape(-1, 1)
target = 2 * x + 1
losses = []
for step in range(40):
    optimizer.zero_grad()
    loss = ((model(x) - target) ** 2).mean()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print("初始/最终损失:", losses[0], losses[-1])
print("学到的参数:", model.weight.item(), model.bias.item())
assert losses[-1] < losses[0]


初始/最终损失: 0.570510745048523 0.0017381682991981506
学到的参数: 1.9349778890609741 0.9999774098396301


## 9. FLOPs 与资源估算

矩阵 [M,K] × [K,N] 的常用计算量估算约 2MKN。训练6ND是近似，不包括所有架构、优化器与通信开销。

In [10]:
M, K, N = 32, 128, 64
print("矩阵乘法估算 FLOPs:", 2 * M * K * N)
parameters, training_tokens = 70e9, 15e12
gpu_count, peak_flops, mfu = 1024, 1979e12 / 2, 0.5
days = 6 * parameters * training_tokens / (gpu_count * peak_flops * mfu) / 86400
print("采用课堂假设估算训练天数:", days)


矩阵乘法估算 FLOPs: 524288
采用课堂假设估算训练天数: 143.92659381842682


## 10. 梯度累积

比较一个大 batch 与两个等大小 microbatch 的梯度。平均损失按 microbatch 数缩放；不涉及 BatchNorm 或 dropout。

In [11]:
torch.manual_seed(7)
x, target = torch.randn(8, 3), torch.randn(8, 1)
model = nn.Linear(3, 1, bias=False)
model.zero_grad()
((model(x) - target) ** 2).mean().backward()
reference_gradient = model.weight.grad.clone()
model.zero_grad()
for xb, yb in zip(x.chunk(2), target.chunk(2)):
    (((model(xb) - yb) ** 2).mean() / 2).backward()
torch.testing.assert_close(model.weight.grad, reference_gradient)
print("两种方式的梯度一致")


两种方式的梯度一致


## 11. Activation checkpointing

概念演示：用反向时重新计算部分前向结果换取较少的激活保存。小CPU例子仅验证梯度一致，不测显存节省。

In [12]:
from torch.utils.checkpoint import checkpoint
torch.manual_seed(7)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
x = torch.randn(3, 4, requires_grad=True)
model.zero_grad()
model(x).square().mean().backward()
reference = [p.grad.clone() for p in model.parameters()]
model.zero_grad()
x.grad = None
checkpoint(model, x, use_reentrant=False).square().mean().backward()
for p, expected in zip(model.parameters(), reference):
    torch.testing.assert_close(p.grad, expected)
print("checkpoint 前后参数梯度一致")


checkpoint 前后参数梯度一致


## 下一步

本周 C3 优先完成1～7节并能解释形状。8～11节是训练与资源背景，可稍后补。GPU性能、roofline和课堂大规模示例请阅读录课版，不以本练习的CPU时间代替原课件GPU性能。